In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from collections import Counter

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
train_dir = r'C:\Users\sagal\Desktop\Let us build\RAF-DB\DATASET\train'

# Seeds
random.seed(42)
torch.manual_seed(42)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2))
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=train_dir, transform=val_transform)

# Split and Sampler
indices = list(range(len(train_dataset)))
random.seed(42)
random.shuffle(indices)

split_size    = int(0.85 * len(indices))
train_indices = indices[:split_size]
val_indices   = indices[split_size:]

train_subset = Subset(train_dataset, train_indices)
val_subset   = Subset(val_dataset, val_indices)

train_labels = [train_dataset.targets[i] for i in train_indices]
class_counts = Counter(train_labels)
total = len(train_labels)
class_weights = {cls: total / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_subset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_subset, batch_size=32, shuffle=False)

In [3]:
# Load ResNet-50
resnet = models.resnet50(weights="IMAGENET1K_V2")

# Unfreeze layer3, layer4, and fc
for name, param in resnet.named_parameters():
    if "layer3" in name or "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Replace head (ResNet-50 in_features is 2048 instead of 512)
resnet.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(resnet.fc.in_features, 7)
)

model = resnet.to(device)

# Quick verification
dummy = torch.randn(32, 3, 224, 224).to(device)
print("Output shape:", model(dummy).shape)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\sagal/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 54.3MB/s]


Output shape: torch.Size([32, 7])


In [4]:
os.makedirs("../../models", exist_ok=True)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = Adam([
    {"params": resnet.layer3.parameters(), "lr": 1e-6, "weight_decay": 1e-4},
    {"params": resnet.layer4.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
    {"params": resnet.fc.parameters(),     "lr": 1e-4, "weight_decay": 1e-3}
])

num_epochs       = 40
patience         = 10
best_val_loss    = float("inf")
patience_counter = 0

save_path = "../../models/best_rafdb_resnet50.pth"

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), save_path)
        print(f"  ✓ Best model saved to {save_path} (val loss: {val_loss:.4f}, val acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(save_path))
print("Best ResNet-50 model loaded!")

Epoch [1/40]  Train Loss: 1.9104  Train Acc: 23.14%  |  Val Loss: 1.8919  Val Acc: 22.65%
  ✓ Best model saved to ../../models/best_rafdb_resnet50.pth (val loss: 1.8919, val acc: 22.65%)
Epoch [2/40]  Train Loss: 1.7876  Train Acc: 31.88%  |  Val Loss: 1.7767  Val Acc: 32.43%
  ✓ Best model saved to ../../models/best_rafdb_resnet50.pth (val loss: 1.7767, val acc: 32.43%)
Epoch [3/40]  Train Loss: 1.6611  Train Acc: 38.81%  |  Val Loss: 1.7265  Val Acc: 35.36%
  ✓ Best model saved to ../../models/best_rafdb_resnet50.pth (val loss: 1.7265, val acc: 35.36%)
Epoch [4/40]  Train Loss: 1.5945  Train Acc: 42.37%  |  Val Loss: 1.6410  Val Acc: 40.36%
  ✓ Best model saved to ../../models/best_rafdb_resnet50.pth (val loss: 1.6410, val acc: 40.36%)
Epoch [5/40]  Train Loss: 1.5337  Train Acc: 45.43%  |  Val Loss: 1.5738  Val Acc: 44.38%
  ✓ Best model saved to ../../models/best_rafdb_resnet50.pth (val loss: 1.5738, val acc: 44.38%)
Epoch [6/40]  Train Loss: 1.4793  Train Acc: 49.00%  |  Val Loss: